In [1]:
"""
generate_html.py  —  reads sw_solution.json, writes visual.html
Four panels:
  1. Output in levels (NK + trend, flex-price y*, HP trend)
  2. Output gap (three concepts: y-y*, y-HP, y-trend)
  3. Inflation (annualised, level with target baseline)
  4. Nominal rate + natural rate (annualised, levels)
"""

import json, math

with open("sw_solution.json") as f:
    sol = json.load(f)

# ── Build JS-ready data blob ───────────────────────────────────────────────
# ghx/ghu come from MATLAB as list-of-rows already
ghx         = sol["ghx"]          # n_endo x n_states
ghu         = sol["ghu"]          # n_endo x n_exo
state_rows  = sol["state_rows"]   # 0-indexed DR rows that are state vars
VI          = sol["VI"]           # 0-indexed DR positions of display vars
shock_keys  = sol["shock_keys"]
shock_labels= sol["shock_labels"]
shock_idx   = sol["shock_js_idx"] # 0-indexed column in ghu
model_rhos  = sol["model_rhos"]
p           = sol["params"]

shocks_js = [
    {"key": shock_keys[i], "label": shock_labels[i],
     "idx": shock_idx[i],  "rho": model_rhos[i]}
    for i in range(len(shock_keys))
]

data = {
    "ghx":        ghx,
    "ghu":        ghu,
    "state_rows": state_rows,
    "VI":         VI,
    "shocks":     shocks_js,
    "params":     p,
}
data_json = json.dumps(data, separators=(",",":"))

ctrend     = p["ctrend"]      # % per quarter
constepinf = p["constepinf"]  # % per quarter

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>SW2007 — IRF Explorer</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*,*::before,*::after{{box-sizing:border-box;margin:0;padding:0}}
:root{{
  --bg:#fff;--bg2:#f5f4f0;--bg3:#eeecea;
  --border:rgba(0,0,0,.09);--border2:rgba(0,0,0,.17);
  --text:#1a1a18;--text2:#5f5e5a;--text3:#888780;
  --blue:#378ADD;--teal:#1D9E75;--coral:#D85A30;--amber:#BA7517;
  --r:8px;--rl:12px;
}}
@media(prefers-color-scheme:dark){{
  :root{{--bg:#1c1c1a;--bg2:#252523;--bg3:#2e2e2b;
        --border:rgba(255,255,255,.09);--border2:rgba(255,255,255,.17);
        --text:#e8e6df;--text2:#b4b2a9;--text3:#888780}}
}}
body{{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
      background:var(--bg3);color:var(--text);font-size:14px;line-height:1.5}}
.page{{max-width:1140px;margin:0 auto;padding:2rem 1.25rem 4rem}}
h1{{font-size:19px;font-weight:500;margin-bottom:3px}}
.sub{{font-size:12px;color:var(--text2);margin-bottom:1.4rem}}
.card{{background:var(--bg);border:.5px solid var(--border);
       border-radius:var(--rl);padding:1.1rem 1.25rem;margin-bottom:.9rem}}
.card-title{{font-size:11px;font-weight:500;color:var(--text3);
             text-transform:uppercase;letter-spacing:.06em;margin-bottom:.85rem}}

/* Shock grid */
.shock-grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(240px,1fr));gap:10px}}
.shock-block{{background:var(--bg2);border-radius:var(--r);padding:10px 12px}}
.shock-head{{display:flex;align-items:center;gap:10px;margin-bottom:8px}}
.tog{{position:relative;width:34px;height:19px;flex-shrink:0;cursor:pointer}}
.tog input{{opacity:0;width:0;height:0}}
.tog-track{{position:absolute;inset:0;background:var(--border2);border-radius:10px;transition:.18s}}
.tog input:checked+.tog-track{{background:var(--blue)}}
.tog-thumb{{position:absolute;width:13px;height:13px;background:#fff;
            border-radius:50%;top:3px;left:3px;transition:.18s}}
.tog input:checked~.tog-thumb{{left:18px}}
.shock-name{{flex:1;font-size:13px}}
.shock-val{{font-size:12px;color:var(--text2);font-variant-numeric:tabular-nums}}
.sl-row{{display:flex;justify-content:space-between;font-size:11px;
         color:var(--text3);margin-bottom:3px}}
input[type=range]{{width:100%;-webkit-appearance:none;height:3px;border-radius:2px;
                   background:var(--bg3);outline:none;cursor:pointer;margin-bottom:7px}}
input[type=range]::-webkit-slider-thumb{{-webkit-appearance:none;width:14px;height:14px;
  border-radius:50%;background:var(--blue);border:2px solid var(--bg)}}
input[type=range]::-moz-range-thumb{{width:12px;height:12px;border-radius:50%;
  background:var(--blue);border:2px solid var(--bg)}}

/* Settings row */
.settings{{display:flex;gap:10px;flex-wrap:wrap;margin-bottom:.9rem}}
.ctrl{{background:var(--bg2);border-radius:var(--r);padding:9px 12px;min-width:160px}}
.ctrl-head{{display:flex;justify-content:space-between;margin-bottom:6px}}
.ctrl-name{{font-size:12px;color:var(--text2)}}
.ctrl-val{{font-size:14px;font-weight:500;font-variant-numeric:tabular-nums}}

/* Legend */
.legend{{display:flex;gap:16px;flex-wrap:wrap;font-size:12px;color:var(--text2);margin-bottom:.85rem}}
.leg{{display:flex;align-items:center;gap:5px}}
.leg-line{{width:20px;height:2px;border-radius:1px;flex-shrink:0}}
.leg-line.dash{{background:none!important;border-top:2px dashed}}
.leg-line.dot{{background:none!important;border-top:2px dotted}}

/* Chart grid — 2x2 */
.chart-grid{{display:grid;grid-template-columns:1fr 1fr;gap:12px}}
.chart-card{{background:var(--bg);border:.5px solid var(--border);
             border-radius:var(--rl);padding:12px 14px}}
.chart-name{{font-size:13px;font-weight:500;margin-bottom:2px}}
.chart-sub{{font-size:11px;color:var(--text3);margin-bottom:8px}}
.chart-wrap{{position:relative;height:220px}}
.eq{{font-size:11px;color:var(--text3);line-height:1.9}}
@media(max-width:650px){{
  .chart-grid{{grid-template-columns:1fr}}
  .page{{padding:1rem}}
}}
</style>
</head>
<body>
<div class="page">

<h1>Smets-Wouters (2007) — Impulse Response Explorer</h1>
<p class="sub">Select any combination of shocks. IRFs are log-deviations from balanced growth path.
Trend ({ctrend:.3f}%/qtr, {ctrend*4:.2f}%/yr) and inflation target ({constepinf:.2f}%/qtr, {constepinf*4:.1f}%/yr) added to level charts.</p>

<div class="card">
  <div class="card-title">Shocks — select any combination</div>
  <div class="shock-grid" id="shock-grid"></div>
</div>

<div class="settings">
  <div class="ctrl">
    <div class="ctrl-head">
      <span class="ctrl-name">Horizon (quarters)</span>
      <span class="ctrl-val" id="v-T">40</span>
    </div>
    <input type="range" id="sl-T" min="20" max="80" step="4" value="40" oninput="update()">
  </div>
</div>

<div class="legend">
  <div class="leg"><div class="leg-line" style="background:var(--blue)"></div>NK output / variable</div>
  <div class="leg"><div class="leg-line dash" style="border-color:var(--teal)"></div>Flex-price output (y*)</div>
  <div class="leg"><div class="leg-line dot" style="border-color:var(--coral)"></div>HP trend</div>
  <div class="leg"><div class="leg-line dash" style="border-color:var(--amber)"></div>Natural rate / target</div>
</div>

<div class="chart-grid">

  <div class="chart-card">
    <div class="chart-name">Output (log level, % from t=0)</div>
    <div class="chart-sub">NK output + deterministic trend; flex-price y* dashed; HP trend dotted</div>
    <div class="chart-wrap"><canvas id="c-y" aria-label="Output">Output</canvas></div>
  </div>

  <div class="chart-card">
    <div class="chart-name">Output gap (%)</div>
    <div class="chart-sub">Three concepts: y minus flex-price y*, minus HP trend, minus linear trend</div>
    <div class="chart-wrap"><canvas id="c-gap" aria-label="Output gap">Gap</canvas></div>
  </div>

  <div class="chart-card">
    <div class="chart-name">Inflation (%, annualised)</div>
    <div class="chart-sub">Level including {constepinf*4:.1f}% annual target; dashed = target</div>
    <div class="chart-wrap"><canvas id="c-pi" aria-label="Inflation">Inflation</canvas></div>
  </div>

  <div class="chart-card">
    <div class="chart-name">Interest rates (%, annualised)</div>
    <div class="chart-sub">Nominal rate (blue) and natural rate r* (amber) — levels</div>
    <div class="chart-wrap"><canvas id="c-r" aria-label="Rates">Rates</canvas></div>
  </div>

</div>

<div class="card" style="margin-top:.9rem">
  <div class="card-title">Model</div>
  <div class="eq">
    Smets &amp; Wouters (2007) — medium-scale NK DSGE · Capital accumulation · Investment adjustment costs · Capital utilization<br>
    Habits λ={p['chabb']:.2f} · σ={p['csigma']:.2f} · Calvo prices ξ_p={p['cprobp']:.2f} · Calvo wages ξ_w={p['cprobw']:.2f}<br>
    Taylor: ρ={p['crr']:.3f} · φ_π={p['crpi']:.3f} · φ_y={p['cry']:.4f} · φ_Δy={p['crdy']:.4f}<br>
    Solution: Dynare first-order perturbation · Output gap = NK output − flex-price output · HP filter λ=1600
  </div>
</div>

</div>

<script>
const D = {data_json};
const GHX = D.ghx, GHU = D.ghu;
const SR  = D.state_rows;   // 0-indexed DR rows that are states
const VI  = D.VI;           // 0-indexed DR positions of display vars
const P   = D.params;
const TREND_Q = P.ctrend;        // % per quarter
const PI_TGT  = P.constepinf;   // % per quarter

// ── HP filter ──────────────────────────────────────────────────────────────
function hpFilter(y, lam=1600) {{
  const n = y.length;
  // Build D'D (pentadiagonal) then solve (I + lam*D'D)*trend = y
  const a = new Array(n).fill(0), b = new Array(n).fill(0),
        c = new Array(n).fill(0), d = new Array(n).fill(0),
        e = new Array(n).fill(0);
  // D'D diagonals
  const dd = [1,-4,6,-4,1];
  for (let i=2;i<n-2;i++) {{
    a[i]+=lam*1; b[i]+=lam*-4; c[i]+=lam*6; d[i]+=lam*-4; e[i]+=lam*1;
  }}
  if(n>2){{ c[0]+=lam*1;d[0]+=lam*-2;e[0]+=lam*1; }}
  if(n>3){{ b[1]+=lam*-2;c[1]+=lam*5;d[1]+=lam*-4;e[1]+=lam*1; }}
  if(n>3){{ a[n-2]+=lam*1;b[n-2]+=lam*-4;c[n-2]+=lam*5;d[n-2]+=lam*-2; }}
  if(n>2){{ a[n-1]+=lam*1;b[n-1]+=lam*-2;c[n-1]+=lam*1; }}
  for(let i=0;i<n;i++) c[i]+=1; // + I

  // Sparse Gaussian elimination for pentadiagonal system
  // Use dense fallback for small T (max 80 here)
  const A = Array.from({{length:n}},()=>new Array(n).fill(0));
  for(let i=0;i<n;i++){{
    if(i-2>=0) A[i][i-2]=a[i];
    if(i-1>=0) A[i][i-1]=b[i];
    A[i][i]=c[i];
    if(i+1<n)  A[i][i+1]=d[i];
    if(i+2<n)  A[i][i+2]=e[i];
  }}
  const rhs=[...y];
  for(let col=0;col<n;col++){{
    let mx=col;
    for(let r=col+1;r<n;r++) if(Math.abs(A[r][col])>Math.abs(A[mx][col])) mx=r;
    [A[col],A[mx]]=[A[mx],A[col]]; [rhs[col],rhs[mx]]=[rhs[mx],rhs[col]];
    const pv=A[col][col]; if(Math.abs(pv)<1e-14) continue;
    for(let r=col+1;r<n;r++){{
      const f=A[r][col]/pv;
      for(let k=col;k<n;k++) A[r][k]-=f*A[col][k];
      rhs[r]-=f*rhs[col];
    }}
  }}
  const tr=new Array(n).fill(0);
  for(let i=n-1;i>=0;i--){{
    let s=rhs[i];
    for(let j=i+1;j<n;j++) s-=A[i][j]*tr[j];
    tr[i]=s/A[i][i];
  }}
  return tr;
}}

// ── IRF propagation ────────────────────────────────────────────────────────
function propagate(shock_col, size, T) {{
  // shock_col: 0-indexed column in GHU
  const nV = GHX.length;
  const path = Array.from({{length:T}},()=>new Array(nV).fill(0));
  for(let v=0;v<nV;v++) path[0][v] = GHU[v][shock_col]*size;
  for(let t=1;t<T;t++){{
    const sv = SR.map(r=>path[t-1][r]);
    for(let v=0;v<nV;v++){{
      let s=0;
      for(let k=0;k<sv.length;k++) s+=GHX[v][k]*sv[k];
      path[t][v]=s;
    }}
  }}
  return path;
}}

// Combine shocks (linear superposition + user persistence scaling)
function computeIRF(entries, T) {{
  const nV = GHX.length;
  const combined = Array.from({{length:T}},()=>new Array(nV).fill(0));
  for(const e of entries){{
    const base = propagate(e.shockIdx, e.size, T);
    const mr = e.modelRho || 0.01;
    const ur = e.rho;
    for(let t=0;t<T;t++){{
      const scale = t===0 ? 1.0 : Math.pow(ur/Math.max(mr,0.001), t);
      for(let v=0;v<nV;v++) combined[t][v]+=base[t][v]*scale;
    }}
  }}
  return combined;
}}

// ── Chart factory ──────────────────────────────────────────────────────────
const dark=()=>window.matchMedia('(prefers-color-scheme:dark)').matches;
const gc=()=>dark()?'rgba(255,255,255,.07)':'rgba(0,0,0,.06)';
const tc=()=>dark()?'#777':'#999';
const zc=()=>dark()?'rgba(255,255,255,.2)':'rgba(0,0,0,.15)';

function ds(data,color,dash=[],width=1.8,label=''){{
  return{{data,label,borderColor:color,borderWidth:width,
          borderDash:dash,pointRadius:0,tension:0.3,fill:false}};
}}

function mkChart(id,datasets){{
  return new Chart(document.getElementById(id),{{
    type:'line', data:{{labels:[],datasets}},
    options:{{
      responsive:true,maintainAspectRatio:false,animation:{{duration:180}},
      interaction:{{mode:'index',intersect:false}},
      plugins:{{legend:{{display:false}},
               tooltip:{{callbacks:{{label:c=>`${{c.dataset.label||''}}: ${{c.parsed.y.toFixed(3)}}`}}}}}},
      scales:{{
        x:{{grid:{{color:gc()}},ticks:{{color:tc(),font:{{size:10}},maxTicksLimit:9}},
            title:{{display:true,text:'quarters',color:tc(),font:{{size:10}}}}}},
        y:{{grid:{{color:ctx=>ctx.tick.value===0?zc():gc(),
                  lineWidth:ctx=>ctx.tick.value===0?1.5:1}},
            ticks:{{color:tc(),font:{{size:10}},callback:v=>v.toFixed(2)}}}}
      }}
    }}
  }});
}}

const BL='#378ADD',TE='#1D9E75',CO='#D85A30',AM='#BA7517';

const charts={{
  y:   mkChart('c-y',  [ds([],BL,[],1.9,'NK output'),
                         ds([],TE,[6,4],1.4,'Flex y*'),
                         ds([],CO,[3,3],1.2,'HP trend')]),
  gap: mkChart('c-gap',[ds([],BL,[],1.9,'Gap: y − y*'),
                         ds([],CO,[4,3],1.4,'Gap: y − HP'),
                         ds([],AM,[6,4],1.3,'Gap: y − trend')]),
  pi:  mkChart('c-pi', [ds([],BL,[],1.9,'Inflation'),
                         ds([],AM,[6,4],1.3,'Target')]),
  r:   mkChart('c-r',  [ds([],BL,[],1.9,'Nominal rate'),
                         ds([],AM,[6,4],1.4,'Natural rate r*')]),
}};

// ── Shock UI ───────────────────────────────────────────────────────────────
const shockState={{}};
D.shocks.forEach(sh=>{{
  shockState[sh.key]={{active:false,size:1.0,rho:sh.rho}};
}});

function buildGrid(){{
  const grid=document.getElementById('shock-grid');
  D.shocks.forEach(sh=>{{
    const block=document.createElement('div');
    block.className='shock-block';
    block.innerHTML=`
      <div class="shock-head">
        <label class="tog">
          <input type="checkbox" id="chk-${{sh.key}}" onchange="toggleShock('${{sh.key}}')">
          <div class="tog-track"></div><div class="tog-thumb"></div>
        </label>
        <span class="shock-name">${{sh.label}}</span>
        <span class="shock-val" id="sv-${{sh.key}}">1.0σ</span>
      </div>
      <div class="sl-row"><span>Size (σ)</span><span id="sv2-${{sh.key}}">1.0</span></div>
      <input type="range" min="0.1" max="3" step="0.1" value="1"
             oninput="setSz('${{sh.key}}',this.value)">
      <div class="sl-row"><span>Persistence ρ</span><span id="rv-${{sh.key}}">${{sh.rho.toFixed(2)}}</span></div>
      <input type="range" min="0" max="0.98" step="0.01" value="${{sh.rho}}"
             oninput="setRho('${{sh.key}}',this.value)">
    `;
    grid.appendChild(block);
  }});
}}

function toggleShock(k){{shockState[k].active=document.getElementById('chk-'+k).checked;update();}}
function setSz(k,v){{shockState[k].size=+v;document.getElementById('sv-'+k).textContent=parseFloat(v).toFixed(1)+'σ';document.getElementById('sv2-'+k).textContent=parseFloat(v).toFixed(1);update();}}
function setRho(k,v){{shockState[k].rho=+v;document.getElementById('rv-'+k).textContent=parseFloat(v).toFixed(2);update();}}

// ── Main update ────────────────────────────────────────────────────────────
function update(){{
  const T=+document.getElementById('sl-T').value;
  document.getElementById('v-T').textContent=T;
  const Q=Array.from({{length:T}},(_,i)=>i);

  const active=D.shocks
    .filter(sh=>shockState[sh.key].active)
    .map(sh=>{{return{{
      shockIdx:sh.idx, size:shockState[sh.key].size,
      rho:shockState[sh.key].rho, modelRho:sh.rho, key:sh.key
    }}}});

  // Zero state if nothing selected
  const zeros=new Array(T).fill(0);
  if(!active.length){{
    // trend path still shows even with no shocks
    const tr=Q.map(t=>TREND_Q*t);
    const hp=hpFilter(tr);
    setChart(charts.y,Q,[tr,tr,hp]);
    setChart(charts.gap,Q,[zeros,zeros,zeros]);
    setChart(charts.pi,Q,[zeros.map(()=>PI_TGT*4),zeros.map(()=>PI_TGT*4)]);
    setChart(charts.r,Q,[zeros,zeros]);
    return;
  }}

  const path=computeIRF(active,T);
  const get=vi=>path.map(s=>s[vi]);

  const y_dev  =get(VI.y);
  const yf_dev =get(VI.yf);
  const pi_dev =get(VI.pi);
  const r_dev  =get(VI.r);
  const rrf_dev=get(VI.rrf);

  // Level paths (log-deviation + deterministic trend)
  const trend  =Q.map(t=>TREND_Q*t);
  const y_lev  =y_dev.map((v,t)=>v+trend[t]);
  const yf_lev =yf_dev.map((v,t)=>v+trend[t]);
  const hp     =hpFilter(y_lev);

  // Three output gaps (all in %)
  const gap_flex =y_dev.map((v,i)=>v-yf_dev[i]);
  const gap_hp   =y_lev.map((v,i)=>v-hp[i]);
  const gap_trend=y_dev;   // deviation from trend is just y_dev itself

  // Inflation annualised level
  const pi_ann  =pi_dev.map(v=>v*4+PI_TGT*4);
  const tgt_line=zeros.map(()=>PI_TGT*4);

  // Rates annualised — need SS nominal rate to show level
  // SS nominal rate ≈ (r* + pi_target) annualised; deviations are r_dev and rrf_dev
  const r_ss = (1/P.crr - 1 + PI_TGT) * 4;  // rough SS approximation
  const r_ann  =r_dev.map(v=>v*4);
  const rrf_ann=rrf_dev.map(v=>v*4);

  setChart(charts.y,  Q,[y_lev, yf_lev, hp]);
  setChart(charts.gap,Q,[gap_flex, gap_hp, gap_trend]);
  setChart(charts.pi, Q,[pi_ann, tgt_line]);
  setChart(charts.r,  Q,[r_ann, rrf_ann]);
}}

function setChart(ch,labels,series){{
  ch.data.labels=labels;
  series.forEach((s,i)=>{{ch.data.datasets[i].data=s;}});
  ch.update('none');
}}

document.getElementById('sl-T').addEventListener('input',update);
buildGrid();
update();
</script>
</body>
</html>"""

with open("visual.html","w",encoding="utf-8") as f:
    f.write(html)
print("visual.html written")

visual.html written
